In [ ]:
#This is an example to make embeddings for a neo4j graph


from openai import OpenAI
from neo4j import GraphDatabase

# URI examples: "neo4j://localhost", "neo4j+s://xxx.databases.neo4j.io"
URI='neo4j+s://eo-neo4j-tst.us.lmco.com/browser/'
AUTH = ("USERNAME", "PASSWORD")

with GraphDatabase.driver(URI, auth=AUTH) as driver:
    driver.verify_connectivity()
driver=GraphDatabase.driver(URI, auth=AUTH)
#get neo4j node-initial driver first
plots=[]
db='eo.recommendations50' #this is an example dtabase that has many movies and actors etc in it
records, summary, keys = driver.execute_query("""
MATCH (n:Movie) 
WHERE n.plot IS NOT NULL
RETURN n 
limit 50
    """,
   database_=db,
)

#make list of concatenated properties for each node you want to make the embedding for
nodes=[]
for record in records:
    data_rec=record.data()
    plots.append(data_rec['n']['plot'])
    nodes.append( {'plot':data_rec['n']['plot'],'title':data_rec['n']['title'] } )
d=data_rec


# Define the maximum batch size based on API limits
BATCH_SIZE = 200

api_key ='KEY'
base_url='https://api.ai.us.lmco.com/v1'
client = OpenAI(
                api_key=api_key, base_url=base_url
            )
all_embeddings = []

# Process the data in batches to make embeddings
for i in range(0, len(plots), BATCH_SIZE):
    print(i)
    batch = plots[i:i + BATCH_SIZE]

    # Call the embeddings API for the current batch
    response = client.embeddings.create(
                model="mxbai-embed-large-v1", input=batch
            )

    # Extract the embeddings from the response
    batch_embeddings = [item.embedding for item in response.data]
    all_embeddings.extend(batch_embeddings)

#place embeddings in list with other node data - this will be used to write to graph 
for i, article in enumerate(nodes):
    #article['embedding'] = response_dict['data'][i]['embedding']
    article['embedding'] = all_embeddings[i]
#print(nodes)



0
[{'plot': "A cowboy doll is profoundly threatened and jealous when a new spaceman figure supplants him as top toy in a boy's room.", 'title': 'Toy Story', 'embedding': [0.031244680285453796, 0.013214013539254665, 0.023292258381843567, 0.03401319682598114, 0.0015519933076575398, -0.05825183540582657, 0.004117461387068033, 0.024323388934135437, 0.00920249056071043, -6.163170473882928e-05, 0.027049530297517776, -0.006818883121013641, -0.030453674495220184, 0.014139206148684025, -0.0023818418849259615, 0.0007671684725210071, -0.03613195940852165, -0.04070848226547241, -0.05118929594755173, 0.007556918542832136, 0.011370690539479256, 0.012373571284115314, -0.045398011803627014, 0.0009843416046351194, 0.02387138642370701, 0.048053525388240814, 0.019026484340429306, 0.011455440893769264, 0.033900193870067596, 0.011398940347135067, 1.8125349015463144e-05, -0.008150171488523483, 0.045030757784843445, 0.0035259732976555824, -0.027939410880208015, -0.02548164688050747, 0.02946491912007332, -0.0

In [4]:
#write embedding to the relevent nodes

cypher = """
UNWIND $nodes_to_make as node
call (node) {
    MATCH (bsp:Movie {title: node.title})
    call db.create.setNodeVectorProperty(bsp, "text_embedding_1", node.embedding)
} in transactions of 1000 rows
"""

db='eo.recommendations50'
driver=GraphDatabase.driver(URI, auth=AUTH)

with driver.session(database=db) as session:
    tp=session.run(
        cypher,
        {"nodes_to_make": nodes},
    )


In [ ]:
#set the embedding property to be a vector index

from neo4j import GraphDatabase

# Establish a connection to your Neo4j database
URI = "URL"  # Replace with your Neo4j URI
URI='neo4j+s://eo-neo4j-tst.us.lmco.com/browser/'
AUTH = ("neo4j", "PASSWORD")  # Replace with your credentials

db='eo.recommendations50'
driver = GraphDatabase.driver(URI, auth=AUTH)
#with GraphDatabase.driver(URI, auth=AUTH) as driver:
    #driver.verify_connectivity()
# Define your vector index parameters
index_name = "my_vector_index_feb10"
node_label = "Movie"  # The label of nodes to be indexed
embedding_property = "text_embedding_1"  # The property containing the vector embeddings
embedding_dimensions = 1536  # The dimension of your embeddings (e.g., for OpenAI)
similarity_function = "COSINE"  # Or EUCLIDEAN, etc.

# Cypher query to create the vector index
create_index_query = f"""
CREATE VECTOR INDEX {index_name} IF NOT EXISTS
FOR (n:{node_label}) ON (n.{embedding_property})
OPTIONS {{indexConfig: {{
  `vector.dimensions`: {embedding_dimensions},
  `vector.similarity_function`: '{similarity_function}'
}}}}
"""

# Execute the Cypher query
with driver.session(database=db) as session:
    session.run(create_index_query)
    print(f"Vector index '{index_name}' created successfully.")

driver.close()

Vector index 'my_vector_index_feb10' created successfully.
